# WiFi基板入れ替え実験 - 個体差の切り分け

同じV5基板（個体1）に異なるWiFi基板を接続し、ESP32モジュールの個体差を確認する。
ESP32プログラムはProbe Request版で統一。

- **v2-1-pr**: 個体1 V5 + 個体1 WiFi基板
- **v2-1-pr-swap**: 個体1 V5 + 個体2 WiFi基板

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Meiryo'
plt.rcParams['axes.unicode_minus'] = False

FILES = {
    '個体1 WiFi基板': 'dist/current_log_all_v2-1-pr.csv',
    '個体2 WiFi基板 (入替)': 'dist/current_log_all_v2-1-pr-swap.csv',
}

SCAN_PATTERNS_MS = [3000, 5000, 10000]
REPS_PER_PATTERN = 5

In [ ]:
# ============================================================
# データ読み込み + WiFi ON区間検出
# ============================================================
results_all = {}

for label, csv_file in FILES.items():
    df = pd.read_csv(csv_file)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    valid_idx = df[df['Current (mA)'] > -10].index[0]
    df = df.iloc[valid_idx:].reset_index(drop=True)
    df['elapsed_s'] = (df['Timestamp'] - df['Timestamp'].iloc[0]).dt.total_seconds()
    
    current = df['Current (mA)'].values
    timestamps = df['elapsed_s'].values
    
    is_on = current > 30
    transitions = np.diff(is_on.astype(int))
    on_starts = np.where(transitions == 1)[0] + 1
    on_ends = np.where(transitions == -1)[0] + 1
    if is_on[0]: on_starts = np.insert(on_starts, 0, 0)
    if is_on[-1]: on_ends = np.append(on_ends, len(is_on))
    n = min(len(on_starts), len(on_ends))
    
    valid = []
    for i in range(n):
        s, e = on_starts[i], on_ends[i]
        dur = timestamps[min(e, len(timestamps)-1)] - timestamps[s]
        if dur >= 1.0:
            valid.append((s, e))
    
    wifi = valid[:len(SCAN_PATTERNS_MS) * REPS_PER_PATTERN]
    
    # ベースライン
    wifi_start_t = timestamps[wifi[0][0]]
    wifi_end_t = timestamps[wifi[-1][1]]
    wifi_on_idx = set()
    for s, e in wifi:
        wifi_on_idx.update(range(s, e))
    wifi_range = (timestamps >= wifi_start_t) & (timestamps <= wifi_end_t)
    wifi_range_idx = set(np.where(wifi_range)[0])
    wifi_off_idx = wifi_range_idx - wifi_on_idx
    wifi_off_mask = df.index.isin(wifi_off_idx)
    baseline = df.loc[wifi_off_mask, 'Current (mA)'].mean()
    
    # パターン別集計
    wifi_results = []
    for i, (s, e) in enumerate(wifi):
        seg = current[s:e]
        dur = timestamps[min(e, len(timestamps)-1)] - timestamps[s]
        pidx = i // REPS_PER_PATTERN
        wifi_results.append({
            'scan_ms': SCAN_PATTERNS_MS[pidx],
            'rep': i % REPS_PER_PATTERN + 1,
            'duration_s': dur,
            'mean_mA': seg.mean() - baseline,
            'peak_mA': seg.max() - baseline,
        })
    
    df_wifi = pd.DataFrame(wifi_results)
    summary = df_wifi.groupby('scan_ms').agg(
        回数=('mean_mA', 'count'),
        ON区間秒=('duration_s', 'mean'),
        平均電流=('mean_mA', 'mean'),
        標準偏差=('mean_mA', 'std'),
    ).reset_index()
    
    results_all[label] = {'summary': summary, 'baseline': baseline, 'df': df, 'wifi': wifi}
    
    print(f'\n=== {label} (baseline={baseline:.2f}mA) ===')
    print(summary.to_string(index=False))

In [ ]:
# ============================================================
# 全体波形の比較 (最初のWiFiスキャンサイクル1つを重ね書き)
# ============================================================
fig, ax = plt.subplots(figsize=(14, 5))

for label, data in results_all.items():
    d = data['df']
    wifi = data['wifi']
    # 最初のON区間の前後を含めて表示
    s = wifi[0][0]
    e = wifi[0][1]
    start_t = d['elapsed_s'].iloc[max(0, s-20)]
    end_t = d['elapsed_s'].iloc[min(len(d)-1, e+40)]
    seg = d[(d['elapsed_s'] >= start_t) & (d['elapsed_s'] <= end_t)]
    ax.plot(seg['elapsed_s'] - start_t, seg['Current (mA)'], linewidth=0.8, label=label, alpha=0.7)

ax.set_xlabel('経過時間 (秒)')
ax.set_ylabel('電流 (mA)')
ax.set_title('WiFi基板入れ替え: 1回のスキャンサイクルの電流波形比較')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 比較グラフ: スキャン時間別の平均電流
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(SCAN_PATTERNS_MS))
width = 0.35
colors = ['steelblue', 'coral']

for i, (label, data) in enumerate(results_all.items()):
    s = data['summary']
    ax.bar(x + i * width, s['平均電流'].values, width, 
           label=label, color=colors[i], alpha=0.8)
    for j, v in enumerate(s['平均電流'].values):
        ax.text(x[j] + i * width, v + 2, f'{v:.1f}', ha='center', fontsize=9)

ax.set_xlabel('スキャン時間')
ax.set_ylabel('平均電流 (mA)')
ax.set_title('WiFi基板入れ替え実験: スキャン時の平均電流比較\n(同一V5基板, Probe Request版)')
ax.set_xticks(x + width / 2)
ax.set_xticklabels([f'{ms}ms' for ms in SCAN_PATTERNS_MS])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 結論
print('=== 結論 ===')
print('同一V5基板でWiFi基板のみ入れ替えた結果、消費電流に大きな差が確認された。')
print('→ 消費電流の差はESP32モジュール（WiFi基板）の個体差に起因する。')